# 05: Data Cleaning (Linked Data)
===========================

**New Workflow: Link First, Then Clean**

This approach uses video context to improve cleaning accuracy:

1. Load linked data (comments + video metadata)
2. Video-level pre-filtering
3. Comment-level cleaning with video context
4. Output high-quality dataset

**Why this order?**
- Video context helps identify relevant comments even if text is short
- Can filter out irrelevant videos before processing comments
- Product categories enable category-specific cleaning rules

In [ ]:
import pandas as pd
import re
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Paths
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "../data"
OUTPUT_DIR = DATA_DIR / "cleaned_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading linked data...")
df = pd.read_parquet(DATA_DIR / "linked_data/comments_video_linked.parquet")
print(f"Original size: {len(df):,} rows")

## Step 1: Video-Level Pre-Filtering

In [ ]:
# 1.1 Remove videos with very few comments
MIN_COMMENTS_PER_VIDEO = 5
video_comment_counts = df.groupby('video_id').size()
valid_videos = video_comment_counts[video_comment_counts >= MIN_COMMENTS_PER_VIDEO].index
df = df[df['video_id'].isin(valid_videos)]
print(f"After removing low-comment videos: {len(df):,} rows")

# 1.2 Remove videos with extremely low engagement
df = df[df['engagement_rate'] >= 1.0]
print(f"After removing low-engagement videos: {len(df):,} rows")

In [ ]:
# 1.3 Define excluded video titles
EXCLUDED_TITLE_PATTERNS = [
    r'\biphone\b', r'\bphone case\b', r'\bsmartphone\b',
    r'\biPad\b', r'\btablet case\b',
    r'\bfashion case\b', r'\bcute case\b', r'\baesthetic case\b',
    r'\bskincare\b', r'\bmakeup\b',
]

def is_excluded_title(title):
    title_lower = str(title).lower()
    return any(re.search(pattern, title_lower) for pattern in EXCLUDED_TITLE_PATTERNS)

df = df[~df['title'].apply(is_excluded_title)]
print(f"After removing excluded video titles: {len(df):,} rows")

# 1.4 Keep only relevant product categories
RELEVANT_CATEGORIES = [
    'camera_optics', 'digital_accessories', 'gaming', 
    'travel_outdoor', 'collectibles', 'tools_equipment',
    'audio', 'general_protection'
]

def has_relevant_category(cats):
    if pd.isna(cats) or cats == 'unknown':
        return False
    return any(c in cats.split('|') for c in RELEVANT_CATEGORIES)

df = df[df['product_categories'].apply(has_relevant_category)]
print(f"After keeping only relevant categories: {len(df):,} rows")

## Step 2: Comment-Level Cleaning

In [ ]:
# 2.1 Remove duplicates
df = df.drop_duplicates(subset=['text_original'])
print(f"After removing duplicates: {len(df):,} rows")

# 2.2 Remove null and very short comments
df = df[df['text_original'].notna()]
df = df[df['text_original'].str.len() > 5]
print(f"After removing empty/very short: {len(df):,} rows")

In [ ]:
# 2.3 Text cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^\w\s!?.,]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text_original'].apply(clean_text)
print(f"Text cleaned for {len(df):,} comments")

In [ ]:
# 2.4 Remove low-value comments
LOW_VALUE_PATTERNS = [
    r'^\s*(lol|lmao|wow|omg|wtf|haha|hahaha|hey|hi|ok|okay|yes|no|thanks?|thx)\s*$',
    r'^(first|second|third)\s*$',
]

def is_low_value(text):
    for pattern in LOW_VALUE_PATTERNS:
        if re.match(pattern, text.lower()):
            return True
    if len(text.split()) < 2:
        return True
    return False

df = df[~df['clean_text'].apply(is_low_value)]
print(f"After removing low-value comments: {len(df):,} rows")

In [ ]:
# 2.5 Filter by product-related keywords
PRODUCT_KEYWORDS = [
    'case', 'cover', 'protect', 'protection', 'damaged', 'damage',
    'drop', 'scratch', 'break', 'broken', 'crack',
    'durable', 'sturdy', 'flimsy', 'cheap', 'quality',
    'storage', 'carry', 'bag', 'box', 'pouch', 'sleeve',
    'packaging', 'unboxing', 'contents', 'included',
    'plastic', 'metal', 'aluminum', 'leather', 'foam', 'padding',
    'rubber', 'nylon', 'fabric', 'hard', 'soft',
    'zipper', 'magnetic', 'clip', 'strap', 'handle', 'pocket',
    'compartment', 'waterproof', 'shockproof', 'lightweight',
    'fits', 'size', 'big', 'small', 'roomy', 'spacious',
    'love', 'hate', 'recommend', 'worth', 'price', 'value',
    'great', 'terrible', 'amazing', 'disappointed', 'impressed',
]

def contains_product_keywords(text):
    return any(kw in text for kw in PRODUCT_KEYWORDS)

df = df[df['clean_text'].apply(contains_product_keywords)]
print(f"After keyword filter: {len(df):,} rows")

## Step 3: Add Demand Signal Indicators

In [ ]:
# ================================================================
# 需求信号定义 - 完整版
# ================================================================

STRONG_DEMAND_PATTERNS = {
    # 购买意图 - 最强信号
    "purchase_intent": [
        r"\bneed\b.*\bcase\b", r"\bneed\b.*\bprotect",
        r"\blooking\b.*\bcase\b", r"\blooking\b.*\bfor\b.*\bprotect",
        r"\bsearching\b.*\bcase\b", r"\bwish\b.*\bhad\b.*\bcase",
        r"\bbuying\b.*\bcase\b", r"\bbought\b.*\bcase\b",
        r"\bjust\s*bought\b", r"\bjust\s*ordered\b", r"\bjust\s*got\b",
        r"can'?t\s*find\b.*\bcase\b", r"nowhere\s*to\s*find\b",
        r"\brecommend\b.*\bcase\b", r"\bany\s+suggestions\b.*\bcase\b",
        r"\bbest\s+case\b.*\bfor\b", r"\bcase\s+recommendation\b",
        r"\bdoes\s+it\s+fit\b.*\bcase\b", r"\bwhat\s+case\s+fits\b",
    ],
    # 问题/抱怨
    "problem_complaint": [
        r"\bbroke\b", r"\bbroken\b", r"\bdamaged\b", r"\bdamage\b",
        r"\bcracked\b", r"\bdropped\b.*\bbroke\b",
        r"\bdoesn'?t\s+protect\b", r"\bdoesn'?t\s+fit\b",
        r"\bcheap\b", r"\bflimsy\b", r"\bpoor\s+quality\b",
        r"\bterrible\b", r"\bworst\b", r"\bdisappointed\b",
        r"\btoo\s+small\b", r"\btoo\s+big\b",
        r"\bneed\s+more\b", r"\bneed\s+better\b",
        r"\bshould\s+have\b.*\bcase\b",
    ],
    # 收纳/携带
    "storage_travel": [
        r"\bhow\s+to\s+pack\b", r"\bpack\s+my\b",
        r"\bpacking\s+list\b", r"\bstorage\s+solution\b",
        r"\borganize\b", r"\borganizer\b",
        r"\bcarry\s+case\b", r"\bcarrying\s+case\b",
        r"\btravel\s+case\b", r"\bEDC\b", r"\beveryday\s+carry\b",
        r"\bportable\b.*\bsolution\b",
        r"\bshockproof\b", r"\bwaterproof\b",
        r"\bhard\s+case\b", r"\bhardshell\b",
    ],
    # 评价/比较
    "review_comparison": [
        r"\bvs\b", r"\bversus\b", r"\bbetter\s+than\b",
        r"\bcompare\s+to\b", r"\binstead\s+of\b",
        r"\breview\b", r"\bunboxing\b",
        r"\bafter\s+using\b", r"\brecommend\b",
        r"\bworth\s+it\b", r"\bprice\s+for\b",
    ],
    # 一般问答
    "general_question": [
        r"\bhow\s+does\b", r"\bwhat\s+is\b", r"\bdoes\s+it\b",
        r"\bcan\s+I\b", r"\bwill\s+it\b",
        r"\bwhere\s+to\s+buy\b", r"\bhow\s+much\b",
    ]
}

EXCLUDE_PATTERNS = {
    "nonsense": [
        r"^\s*(lol|lmao|rofl|wth|omg|wtf|haha|hahaha)\s*$",
        r"^\s*(first|second|third)\s*$",
        r"^\s*(yes|no|ok|okay|yeah)\s*$",
        r"^\s*(hi|hey|yo|sup)\s*$",
        r"^\s*(thanks?|thx|ty)\s*$",
        r"^\s*(nice|cool|awesome|amazing|wow)\s*$",
        r"^\s*(subscribe|like|comment)\s*$",
    ],
    "irrelevant": [
        r"\bphone\s+case\b", r"\biphone\s+case\b",
        r"\bsamsung\s+case\b", r"\biPad\s+case\b",
        r"\bsoftware\b", r"\bdownload\b", r"\bapp\b",
    ]
}

def detect_demand_signals(text):
    text_lower = text.lower()
    signals = []
    for signal_type, patterns in STRONG_DEMAND_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text_lower):
                signals.append(signal_type)
                break
    return signals if signals else ["general"]

def should_exclude_comment(text):
    text_lower = text.lower()
    for patterns in EXCLUDE_PATTERNS.values():
        for pattern in patterns:
            if re.search(pattern, text_lower):
                return True
    return False

def assign_priority(signals):
    if "purchase_intent" in signals or "problem_complaint" in signals:
        return "high"
    if "storage_travel" in signals:
        return "medium"
    return "general"

print(f"Demand patterns: {sum(len(v) for v in STRONG_DEMAND_PATTERNS.values())}")
print(f"Exclude patterns: {sum(len(v) for v in EXCLUDE_PATTERNS.values())}")


In [ ]:
# ================================================================
# Step 3: 需求信号检测
# ================================================================

# 3a: 排除无意义评论
print("Step 3a: Excluding irrelevant comments...")
df["is_excluded"] = df["clean_text"].apply(should_exclude_comment)
excluded_count = df["is_excluded"].sum()
print(f"Excluded: {excluded_count:,} comments")
df = df[~df["is_excluded"]]
print(f"After exclusion: {len(df):,} rows")

# 3b: 检测需求信号
print("
Step 3b: Detecting demand signals...")
df["demand_signals"] = df["clean_text"].apply(detect_demand_signals).apply(lambda x: "|".join(x))
print("
Demand signal distribution:")
print(df["demand_signals"].str.split("|").explode().value_counts())

# 3c: 分配优先级
print("
Step 3c: Assigning priority levels...")
df["priority_level"] = df["demand_signals"].apply(lambda s: assign_priority(s.split("|")))
print("
Priority level distribution:")
print(df["priority_level"].value_counts())


## Step 4: Save Cleaned Dataset

In [ ]:
keep_cols = [
    'comment_id', 'video_id', 'thread_id', 'parent_comment_id', 'is_reply',
    'author_display_name', 'text_original', 'clean_text',
    'comment_like_count', 'published_at',
    'product_categories', 'video_context', 'demand_signals', 'priority_level',
    'title', 'channel_title', 'description',
    'view_count', 'engagement_rate',
    'search_keyword', 'fetched_by'
]
output_cols = [c for c in keep_cols if c in df.columns]
df_clean = df[output_cols].copy()

df_clean.to_parquet(OUTPUT_DIR / "cleaned_comments_linked.parquet", index=False)
print(f"Saved: cleaned_comments_linked.parquet ({len(df_clean):,} rows)")

df_clean.to_csv(OUTPUT_DIR / "cleaned_comments_linked.csv", index=False)
print(f"Saved: cleaned_comments_linked.csv")

## Summary

In [ ]:
print("="*60)
print("CLEANING SUMMARY")
print("="*60)
print(f"\nOriginal (linked): 133,110 rows")
print(f"Cleaned:           {len(df_clean):,} rows")
print(f"Videos covered:   {df_clean['video_id'].nunique():,}")
print(f"\nPriority Levels:")
print(df_clean['priority_level'].value_counts())
print("\n" + "="*60)
print("Ready for NLP Analysis!")
print("="*60)